# Analytical Solutions & Benchmarks in Soft Potato 3.0

Welcome to this interactive tutorial on **closed-form analytical solutions** in **Soft Potato 3.0**.

Analytical solutions provide exact mathematical benchmarks for electrochemical processes. In Soft Potato, they serve two primary purposes:
1. **Validation baselines**: Validating numerical PDE simulation engines (explicit and implicit finite difference methods) against exact solutions.
2. **Fast parameter exploration**: Evaluating peak currents, transient decay curves, and steady-state limits instantly without numerical discretization.

In this notebook, we explore three classic analytical benchmarks:
- **Randles–Sevcik equation**: Peak current in reversible cyclic voltammetry at planar macroelectrodes.
- **Cottrell equation**: Current-time transient following a diffusion-limited potential step.
- **Saito equation**: Steady-state limiting current at an inlaid microdisc electrode.

## 1. Units and Conventions (CGS System)

Soft Potato strictly enforces the **CGS unit system** across all modules:
- **Diffusion coefficient** ($D$): $\text{cm}^2/\text{s}$ ($10^{-5}\text{ cm}^2/\text{s}$ is typical for small ions in water).
- **Bulk concentration** ($c_{\text{bulk}}$): $\text{mol}/\text{cm}^3$ ($1\text{ mM} = 10^{-3}\text{ mol/L} = 10^{-6}\text{ mol/cm}^3$).
- **Electrode area** ($A$): $\text{cm}^2$.
- **Microelectrode radius** ($r$): $\text{cm}$ ($10\,\mu\text{m} = 10^{-3}\text{ cm}$). 
- **Temperature** ($T$): $298.15\text{ K}$ (default internally).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Import analytical equations directly from the Soft Potato facade
from softpotato.analytical import randles_sevcik, cottrell, steady_state_microdisc

## 2. Setting Parameters & Vectorized Calculations

Soft Potato analytical functions are fully vectorized and accept NumPy arrays directly.

In [ ]:
# --- Parameters (Strict CGS Units Enforced by Soft Potato) ---
n_electrons = 1
D_O = 1e-5            # Diffusion coefficient (cm^2/s)
C_bulk = 1e-6         # Bulk concentration (mol/cm^3) -> 1 mM
area = 0.0707         # Planar macroelectrode area (cm^2)

# --- Independent Variable Arrays (Vectorized) ---
v_array = np.linspace(0.01, 1.0, 200)      # Scan rate (V/s)
t_array = np.linspace(0.001, 5.0, 500)     # Time (s)
r_array = np.linspace(1e-4, 25e-4, 200)    # Microdisc radius (cm) (1 um to 25 um)

# --- Compute Analytical Solutions ---
i_p = randles_sevcik(n=n_electrons, area=area, D=D_O, c_bulk=C_bulk, scan_rate=v_array)
i_t = cottrell(t=t_array, n=n_electrons, area=area, D=D_O, c_bulk=C_bulk)
i_ss = steady_state_microdisc(n=n_electrons, radius=r_array, D=D_O, c_bulk=C_bulk)

## 3. Visualization

Below we plot:
1. Peak current $i_p$ versus $\nu^{1/2}$ demonstrating the linear Randles-Sevcik relationship.
2. Transient current decay $i(t)$ versus $t$ showing $t^{-1/2}$ Cottrell behavior.
3. Microdisc steady-state current $i_{ss}$ versus microdisc radius $r$ showing linear scaling with radius (Saito equation).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# 1. Randles-Sevcik: i_p vs. sqrt(v)
axes[0].plot(np.sqrt(v_array), i_p * 1e6, color='#1f77b4', lw=2)
axes[0].set_xlabel(r'$\nu^{1/2}$ / (V/s)$^{1/2}$')
axes[0].set_ylabel(r'$i_p$ / $\mu$A')
axes[0].set_title('Randles-Sevcik (Reversible CV)')
axes[0].grid(True, linestyle='--', alpha=0.7)

# 2. Cottrell: i vs. t
axes[1].plot(t_array, i_t * 1e6, color='#d62728', lw=2)
axes[1].set_xlabel('Time / s')
axes[1].set_ylabel(r'$i$ / $\mu$A')
axes[1].set_title('Cottrell (Planar Step)')
axes[1].grid(True, linestyle='--', alpha=0.7)

# 3. Microdisc: i_ss vs. r
# Converting radius to um and current to nA for standard plotting scaling
axes[2].plot(r_array * 1e4, i_ss * 1e9, color='#2ca02c', lw=2)
axes[2].set_xlabel(r'Radius / $\mu$m')
axes[2].set_ylabel(r'$i_{ss}$ / nA')
axes[2].set_title('Saito (Steady-State Microdisc)')
axes[2].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()